# Build the parish page list

Load `data/01_build_site_list/2_parish_list.csv`, generated by `01_build_site_list.ipynb`, and discover parish page URLs. Run this notebook from the project root. Outputs in `data/02_build_page_list/` are used by `03_extract_ministries.ipynb`.


## 0 Environment setup
Install project dependencies with `%pip install -e .` if needed. Load the project-root `.env` without overriding existing environment variables. Discovery does not require an API key or LLM.

Set `HTML_CACHE_DIR` in `.env` to choose the download cache folder. An existing environment variable takes precedence. Individual scraping functions also accept a `cache_dir` override. Existing cache files are not moved automatically.


In [1]:
from dotenv import load_dotenv

_ = load_dotenv(".env", override=False)


In [2]:
from pathlib import Path

import pandas as pd

input_path = Path("data/01_build_site_list/2_parish_list.csv")
parishes_df = pd.read_csv(input_path)

print(f"Loaded {len(parishes_df)} parishes from {input_path}")
parishes_df.head()


Loaded 784 parishes from data\01_parish_list\parish_list.csv


,name,address,site_url,diocese,category,parish_list_url
0,Father Purcell Memorial Center for Exceptional...,"Montgomery, AL",https://fatherpurcell.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
1,Holy Spirit Catholic Parish,"Montgomery, AL",https://holyspiritmgm.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
2,Blessed Francis Xavier Seelos Parish,"31122 US Hwy 31, Spanish Fort, AL, 36527",https://francisxseelos.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
3,Chapel of Our Lady of Bon Secour,"17266 County Road 49 South, Bon Secour, AL, 36511",https://www.stjohnms.com/our-lady-of-bon-secour,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
4,Christ the King Catholic Church,"711 College Avenue, Daphne, AL, 36526",https://ctkdaphne.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder


## 1 Discover parish site pages
For the first `PARISH_LIMIT` rows, check `/sitemap.xml` and sitemap declarations in `/robots.txt`, follow sitemap indexes, and also collect homepage/navigation links. Keep same-site HTTP(S) page URLs, remove fragments, exclude common file downloads, and deduplicate per parish. Query strings and distinct paths are preserved.

Downloads are cached in the folder selected by `HTML_CACHE_DIR` in `.env` (or the environment). Failures are recorded in `discovery_errors_df` so homepage discovery can still work when a sitemap is unavailable. This step discovers URLs; it does not download every discovered page.

Progress lines start with `cache` when all discovery requests use cached files, or `web` when any request needs the web (including failed requests).

Outputs in `data/02_build_page_list/`: `1_site_pages.csv` for discovered pages and `1_discovery_errors.csv` for discovery failures. Each output filename starts with the section number that creates it.


In [3]:
from utils.scraping import get_cache_dir
from utils.page_discovery import discover_site_pages

PARISH_LIMIT = 5
output_dir = Path("data/02_build_page_list")
cache_dir = get_cache_dir()
selected_parishes = parishes_df.head(PARISH_LIMIT)
page_rows = []
error_rows = []

for parish_index, parish in selected_parishes.iterrows():
    site_url = parish["site_url"] if pd.notna(parish["site_url"]) else ""
    fetch_sources = []
    pages, errors = discover_site_pages(site_url, cache_dir, fetch_sources=fetch_sources)
    parish_info = {
        "parish_index": parish_index,
        "name": parish["name"],
        "diocese": parish["diocese"],
        "site_url": site_url,
    }
    page_rows.extend({**parish_info, **page} for page in pages)
    error_rows.extend({**parish_info, **error} for error in errors)
    source = "cache" if fetch_sources and "web" not in fetch_sources else "web"
    print(f"{source} {parish['name']}: {len(pages)} pages, {len(errors)} discovery issues")

parish_columns = ["parish_index", "name", "diocese", "site_url"]
site_pages_df = pd.DataFrame(page_rows, columns=parish_columns + ["page_url", "sources"])
discovery_errors_df = pd.DataFrame(error_rows, columns=parish_columns + ["url", "error"])
output_dir.mkdir(parents=True, exist_ok=True)
site_pages_df.to_csv(output_dir / "1_site_pages.csv", index=False)
discovery_errors_df.to_csv(output_dir / "1_discovery_errors.csv", index=False)
site_pages_df.head(20)


cache Father Purcell Memorial Center for Exceptional Children: 11 pages, 0 discovery issues
cache Holy Spirit Catholic Parish: 52 pages, 0 discovery issues
cache Blessed Francis Xavier Seelos Parish: 479 pages, 0 discovery issues
web Chapel of Our Lady of Bon Secour: 264 pages, 1 discovery issues
cache Christ the King Catholic Church: 859 pages, 0 discovery issues


,parish_index,name,diocese,site_url,page_url,sources
0,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/,"homepage, homepage_link, sitemap"
1,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/admission,"homepage_link, sitemap"
2,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/blog,sitemap
3,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/bulletins,sitemap
4,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/calendar,sitemap
5,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/contact,"homepage_link, sitemap"
6,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/events,sitemap
7,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/history,"homepage_link, sitemap"
8,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/news,sitemap
9,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/photoalbums,sitemap


In [4]:
# Inspect unavailable sitemaps or other requests that need attention.
discovery_errors_df


,parish_index,name,diocese,site_url,url,error
0,3,Chapel of Our Lady of Bon Secour,Archdiocese of Mobile,https://www.stjohnms.com/our-lady-of-bon-secour,https://www.stjohnms.com/our-lady-of-bon-secour,404 Client Error: Not Found for url: https://s...


## 2 Save page URLs for later content extraction
Load the saved `1_site_pages.csv` and save `2_page_content.json` as a JSON array of records containing the parish website URL (`site_url`) and the individual page URL (`page_url`). Page titles and content will be added later; this step does not download the discovered pages.


In [ ]:
from pathlib import Path

import pandas as pd

output_dir = Path("data/02_build_page_list")
site_pages_df = pd.read_csv(output_dir / "1_site_pages.csv", keep_default_na=False)
page_content_df = site_pages_df[["site_url", "page_url"]].copy()
page_content_df.to_json(
    output_dir / "2_page_content.json", orient="records", indent=2, force_ascii=False,
)
print(f"Saved {len(page_content_df)} page URLs to {output_dir / '2_page_content.json'}")
page_content_df.head()
